In [1]:

!pip install -q numpy pandas matplotlib seaborn tqdm


In [3]:
import zipfile
import os

zip_path = "/content/Student folder.zip"

DATA_DIR = "/content/pathsnap_data"

os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [4]:
import os
import glob

# Search for train and student_test folders
train_matches = glob.glob(
    os.path.join(DATA_DIR, "**", "train"),
    recursive=True
)

test_matches = glob.glob(
    os.path.join(DATA_DIR, "**", "student_test"),
    recursive=True
)

# Verify that both folders exist
if not train_matches:
    raise FileNotFoundError("Could not find the train folder!")

if not test_matches:
    raise FileNotFoundError("Could not find the student_test folder!")

TRAIN_DIR = train_matches[0]
STUDENT_TEST_DIR = test_matches[0]

print("Training folder:", TRAIN_DIR)
print("Student test folder:", STUDENT_TEST_DIR)

Training folder: /content/pathsnap_data/Student folder/train
Student test folder: /content/pathsnap_data/Student folder/student_test


In [5]:
# Get all trip folders
train_trips = sorted([
    folder for folder in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, folder))
])

test_trips = sorted([
    folder for folder in os.listdir(STUDENT_TEST_DIR)
    if os.path.isdir(os.path.join(STUDENT_TEST_DIR, folder))
])

print("Number of training trips:", len(train_trips))
print("Number of student test trips:", len(test_trips))

print("\nFirst 5 training trips:")
print(train_trips[:5])

print("\nFirst 5 student test trips:")
print(test_trips[:5])

Number of training trips: 80
Number of student test trips: 10

First 5 training trips:
['00000001', '00000002', '00000003', '00000004', '00000005']

First 5 student test trips:
['00000000', '00000007', '00000009', '00000040', '00000052']


In [6]:
import pandas as pd
import os

def load_raw_trip(partition, trip_id):

    # Select the correct dataset folder
    if partition == "train":
        folder = TRAIN_DIR
    elif partition == "student_test":
        folder = STUDENT_TEST_DIR
    else:
        raise ValueError("Invalid partition")

    trip_path = os.path.join(folder, trip_id)

    # Load GPS observations
    track = pd.read_csv(
        os.path.join(trip_path, f"{trip_id}.track"),
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude", "timestamp"]
    )

    # Load road network nodes
    nodes = pd.read_csv(
        os.path.join(trip_path, f"{trip_id}.nodes"),
        sep=r"\s+",
        header=None,
        names=["longitude", "latitude"]
    )

    # Load directed road connections
    arcs = pd.read_csv(
        os.path.join(trip_path, f"{trip_id}.arcs"),
        sep=r"\s+",
        header=None,
        names=["from_node", "to_node"]
    )

    # Load reference route
    route = pd.read_csv(
        os.path.join(trip_path, f"{trip_id}.route"),
        sep=r"\s+",
        header=None,
        names=["arc_id"]
    )

    return track, nodes, arcs, route

In [7]:
# Select the first training trip
trip_id = train_trips[0]

track, nodes, arcs, route = load_raw_trip("train", trip_id)

print("Trip ID:", trip_id)

print("\nTRACK:")
print(track.head())

print("\nNODES:")
print(nodes.head())

print("\nARCS:")
print(arcs.head())

print("\nROUTE:")
print(route.head())

Trip ID: 00000001

TRACK:
   longitude   latitude  timestamp
0   8.595203  49.844053        1.0
1   8.595294  49.844441        2.0
2   8.595352  49.844883        3.0
3   8.595390  49.845379        4.0
4   8.595587  49.845750        5.0

NODES:
   longitude   latitude
0   8.581679  49.857880
1   8.584637  49.858295
2   8.580173  49.858567
3   8.584392  49.859119
4   8.584935  49.857396

ARCS:
   from_node  to_node
0          0      281
1          0        1
2          0        5
3          1        0
4          1    40750

ROUTE:
   arc_id
0    2641
1   64973
2  133948
3  133949
4  139208


In [8]:
print("TRACK shape:", track.shape)
print("NODES shape:", nodes.shape)
print("ARCS shape:", arcs.shape)
print("ROUTE shape:", route.shape)

TRACK shape: (1315, 3)
NODES shape: (88809, 2)
ARCS shape: (165790, 2)
ROUTE shape: (525, 1)


In [9]:
# Check data types
print("TRACK DATA TYPES")
print(track.dtypes)

print("\nNODES DATA TYPES")
print(nodes.dtypes)

print("\nARCS DATA TYPES")
print(arcs.dtypes)

print("\nROUTE DATA TYPES")
print(route.dtypes)

TRACK DATA TYPES
longitude    float64
latitude     float64
timestamp    float64
dtype: object

NODES DATA TYPES
longitude    float64
latitude     float64
dtype: object

ARCS DATA TYPES
from_node    int64
to_node      int64
dtype: object

ROUTE DATA TYPES
arc_id    int64
dtype: object


In [10]:
def missing_value_report(df, name):

    print(f"\n{name}")
    print("-" * 40)

    print("Missing values:")
    print(df.isnull().sum())

    print("\nTotal missing values:", df.isnull().sum().sum())


missing_value_report(track, "TRACK")
missing_value_report(nodes, "NODES")
missing_value_report(arcs, "ARCS")
missing_value_report(route, "ROUTE")


TRACK
----------------------------------------
Missing values:
longitude    0
latitude     0
timestamp    0
dtype: int64

Total missing values: 0

NODES
----------------------------------------
Missing values:
longitude    0
latitude     0
dtype: int64

Total missing values: 0

ARCS
----------------------------------------
Missing values:
from_node    0
to_node      0
dtype: int64

Total missing values: 0

ROUTE
----------------------------------------
Missing values:
arc_id    0
dtype: int64

Total missing values: 0


In [11]:
import numpy as np

track_valid = (
    np.isfinite(track[["longitude", "latitude", "timestamp"]]).all(axis=1)
    & track["longitude"].between(-180, 180)
    & track["latitude"].between(-90, 90)
)

print("Total GPS observations:", len(track))
print("Valid GPS observations:", track_valid.sum())
print("Invalid GPS observations:", (~track_valid).sum())

Total GPS observations: 1315
Valid GPS observations: 1315
Invalid GPS observations: 0


In [12]:
nodes_valid = (
    np.isfinite(nodes[["longitude", "latitude"]]).all(axis=1)
    & nodes["longitude"].between(-180, 180)
    & nodes["latitude"].between(-90, 90)
)

print("Total road nodes:", len(nodes))
print("Valid road nodes:", nodes_valid.sum())
print("Invalid road nodes:", (~nodes_valid).sum())

Total road nodes: 88809
Valid road nodes: 88809
Invalid road nodes: 0


In [13]:
arc_values = arcs.to_numpy()

arc_valid = (
    np.isfinite(arc_values).all(axis=1)
    & (arc_values >= 0).all(axis=1)
    & (arc_values < len(nodes)).all(axis=1)
    & (arc_values == np.floor(arc_values)).all(axis=1)
)

print("Total arcs:", len(arcs))
print("Valid arcs:", arc_valid.sum())
print("Invalid arcs:", (~arc_valid).sum())

Total arcs: 165790
Valid arcs: 165790
Invalid arcs: 0


In [14]:
route_ids = route["arc_id"].to_numpy()

route_valid = (
    np.isfinite(route_ids)
    & (route_ids >= 0)
    & (route_ids < len(arcs))
    & (route_ids == np.floor(route_ids))
)

# Check whether referenced arcs are themselves valid
safe_route_ids = np.where(
    route_valid,
    route_ids,
    0
).astype(int)

route_valid &= arc_valid[safe_route_ids]

print("Total route entries:", len(route))
print("Valid route entries:", route_valid.sum())
print("Invalid route entries:", (~route_valid).sum())

Total route entries: 525
Valid route entries: 525
Invalid route entries: 0


In [15]:
# Find exact duplicate GPS observations
duplicate_mask = track.duplicated(
    subset=["longitude", "latitude", "timestamp"],
    keep="first"
)

print("Total GPS observations:", len(track))
print("Duplicate observations:", duplicate_mask.sum())
print("Unique observations:", len(track) - duplicate_mask.sum())

Total GPS observations: 1315
Duplicate observations: 0
Unique observations: 1315


In [16]:
# Calculate time difference between consecutive observations
track["time_difference"] = track["timestamp"].diff()

print("Timestamp differences:")
print(track["time_difference"].describe())

# Count zero, negative and positive time differences
print("\nZero time differences:",
      (track["time_difference"] == 0).sum())

print("Negative time differences:",
      (track["time_difference"] < 0).sum())

print("Positive time differences:",
      (track["time_difference"] > 0).sum())

Timestamp differences:
count    1314.000000
mean        1.032725
std         0.446355
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        16.000000
Name: time_difference, dtype: float64

Zero time differences: 0
Negative time differences: 0
Positive time differences: 1314


In [17]:
# Get route arc IDs
route_ids = route["arc_id"].astype(int).to_numpy()

# Get the corresponding directed arcs
route_arcs = arcs.iloc[route_ids].to_numpy()

# Compare destination of current arc
# with source of next arc
connected = (
    route_arcs[:-1, 1] == route_arcs[1:, 0]
)

total_pairs = len(connected)
connected_pairs = connected.sum()
disconnected_pairs = total_pairs - connected_pairs

print("Total consecutive arc pairs:", total_pairs)
print("Connected pairs:", connected_pairs)
print("Disconnected pairs:", disconnected_pairs)

if total_pairs > 0:
    print(
        "Connectivity percentage:",
        round(connected_pairs / total_pairs * 100, 2),
        "%"
    )

Total consecutive arc pairs: 524
Connected pairs: 524
Disconnected pairs: 0
Connectivity percentage: 100.0 %


In [18]:
all_trips = {
    "train": train_trips,
    "student_test": test_trips
}

trip_summary = []

for partition, trip_ids in all_trips.items():

    for trip_id in trip_ids:

        track, nodes, arcs, route = load_raw_trip(
            partition, trip_id
        )

        # Missing values
        missing = (
            track.isnull().sum().sum()
            + nodes.isnull().sum().sum()
            + arcs.isnull().sum().sum()
            + route.isnull().sum().sum()
        )

        # Duplicate GPS observations
        duplicates = track.duplicated(
            subset=["longitude", "latitude", "timestamp"]
        ).sum()

        # Timestamp differences
        time_diff = track["timestamp"].diff()

        non_increasing_timestamps = (
            time_diff <= 0
        ).sum()

        # Route connectivity
        route_ids = route["arc_id"].to_numpy()

        route_arcs = arcs.iloc[
            route_ids.astype(int)
        ].to_numpy()

        connected = (
            route_arcs[:-1, 1] == route_arcs[1:, 0]
        )

        connectivity = (
            connected.mean() * 100
            if len(connected) > 0 else np.nan
        )

        trip_summary.append({
            "partition": partition,
            "trip_id": trip_id,
            "gps_observations": len(track),
            "road_nodes": len(nodes),
            "road_arcs": len(arcs),
            "route_entries": len(route),
            "missing_values": missing,
            "duplicate_gps": duplicates,
            "non_increasing_timestamps": non_increasing_timestamps,
            "route_connectivity_percent": connectivity
        })

trip_summary_df = pd.DataFrame(trip_summary)

print("Preprocessing summary:")
display(trip_summary_df)

Preprocessing summary:


,partition,trip_id,gps_observations,road_nodes,road_arcs,route_entries,missing_values,duplicate_gps,non_increasing_timestamps,route_connectivity_percent
0,train,00000001,1315,88809,165790,525,0,0,0,100.000000
1,train,00000002,7243,7936,16139,262,0,0,0,99.616858
2,train,00000003,821,9138,18028,79,0,0,0,100.000000
3,train,00000004,1064,7350,15499,320,0,0,0,100.000000
4,train,00000005,1045,898,1579,105,0,0,0,100.000000
...,...,...,...,...,...,...,...,...,...,...
85,student_test,00000056,3958,14759,27631,1323,0,0,0,100.000000
86,student_test,00000065,768,6448,13184,145,0,0,0,100.000000
87,student_test,00000076,1486,26635,56027,374,0,0,0,99.731903
88,student_test,00000077,3911,4424,9622,268,0,0,0,100.000000


In [19]:
print("Total trips:", len(trip_summary_df))

print("\nPartition-wise trip counts:")
print(trip_summary_df["partition"].value_counts())

print("\nTotal GPS observations:",
      trip_summary_df["gps_observations"].sum())

print("Total road nodes:",
      trip_summary_df["road_nodes"].sum())

print("Total road arcs:",
      trip_summary_df["road_arcs"].sum())

print("Total route entries:",
      trip_summary_df["route_entries"].sum())

print("\nTotal missing values:",
      trip_summary_df["missing_values"].sum())

print("Total duplicate GPS observations:",
      trip_summary_df["duplicate_gps"].sum())

Total trips: 90

Partition-wise trip counts:
partition
train           80
student_test    10
Name: count, dtype: int64

Total GPS observations: 227577
Total road nodes: 2024174
Total road arcs: 4061945
Total route entries: 34813

Total missing values: 0
Total duplicate GPS observations: 0


In [23]:

all_trips = {
    "train": train_trips,
    "student_test": test_trips
}

trip_summary = []

for partition, trip_ids in all_trips.items():

    for trip_id in trip_ids:

        track, nodes, arcs, route = load_raw_trip(
            partition, trip_id
        )

        # 1. Missing values
        missing = (
            track.isnull().sum().sum()
            + nodes.isnull().sum().sum()
            + arcs.isnull().sum().sum()
            + route.isnull().sum().sum()
        )

        # 2. GPS validation
        track_valid = (
            np.isfinite(
                track[["longitude", "latitude", "timestamp"]]
            ).all(axis=1).to_numpy()
            & track["longitude"].between(-180, 180).to_numpy()
            & track["latitude"].between(-90, 90).to_numpy()
        )

        # 3. Road node validation
        node_valid = (
            np.isfinite(
                nodes[["longitude", "latitude"]]
            ).all(axis=1).to_numpy()
            & nodes["longitude"].between(-180, 180).to_numpy()
            & nodes["latitude"].between(-90, 90).to_numpy()
        )

        # 4. Arc validation
        arc_values = arcs.to_numpy()

        arc_valid = (
            np.isfinite(arc_values).all(axis=1)
            & (arc_values >= 0).all(axis=1)
            & (arc_values < len(nodes)).all(axis=1)
            & (arc_values == np.floor(arc_values)).all(axis=1)
        )

        # Convert valid arc endpoints to safe integer IDs
        safe_arcs = np.where(
            arc_valid[:, None],
            arc_values,
            0
        ).astype(int)

        # Check whether both endpoint nodes are valid
        arc_valid = arc_valid & (
            node_valid[safe_arcs[:, 0]]
            & node_valid[safe_arcs[:, 1]]
        )

        # 5. Route validation
        route_ids = route["arc_id"].to_numpy()

        route_valid = (
            np.isfinite(route_ids)
            & (route_ids >= 0)
            & (route_ids < len(arcs))
            & (route_ids == np.floor(route_ids))
        )

        # Safely access arc validity
        safe_route_ids = np.where(
            route_valid,
            route_ids,
            0
        ).astype(int)

        route_valid = route_valid & arc_valid[safe_route_ids]

        # 6. Duplicate GPS observations
        duplicates = track.duplicated(
            subset=["longitude", "latitude", "timestamp"]
        ).sum()

        # 7. Timestamp anomalies
        time_diff = track["timestamp"].diff()

        non_increasing = (time_diff <= 0).sum()

        # 8. Route connectivity
        valid_positions = np.flatnonzero(route_valid)

        # Only compare consecutive valid route entries
        valid_pairs = np.diff(valid_positions) == 1

        pair_positions = valid_positions[:-1][valid_pairs]

        if len(pair_positions) > 0:

            current_arc_ids = safe_route_ids[pair_positions]
            next_arc_ids = safe_route_ids[pair_positions + 1]

            connected = (
                safe_arcs[current_arc_ids, 1]
                ==
                safe_arcs[next_arc_ids, 0]
            )

            connected_pairs = int(connected.sum())
            eligible_pairs = len(connected)
            disconnected_pairs = eligible_pairs - connected_pairs

            connectivity = (
                connected_pairs / eligible_pairs * 100
            )

        else:
            connected_pairs = 0
            eligible_pairs = 0
            disconnected_pairs = 0
            connectivity = np.nan

        # 9. Store results
        trip_summary.append({
            "partition": partition,
            "trip_id": trip_id,

            "gps_observations": len(track),
            "invalid_gps": int((~track_valid).sum()),

            "road_nodes": len(nodes),
            "invalid_nodes": int((~node_valid).sum()),

            "road_arcs": len(arcs),
            "invalid_arcs": int((~arc_valid).sum()),

            "route_entries": len(route),
            "invalid_route_entries": int((~route_valid).sum()),

            "missing_values": int(missing),
            "duplicate_gps": int(duplicates),
            "non_increasing_timestamps": int(non_increasing),

            "eligible_route_pairs": eligible_pairs,
            "disconnected_route_pairs": disconnected_pairs,

            "route_connectivity_percent": connectivity
        })

# Convert results into a DataFrame
trip_summary_df = pd.DataFrame(trip_summary)

print("Validation completed!")
print("Total trips checked:", len(trip_summary_df))

display(trip_summary_df)


Validation completed!
Total trips checked: 90


,partition,trip_id,gps_observations,invalid_gps,road_nodes,invalid_nodes,road_arcs,invalid_arcs,route_entries,invalid_route_entries,missing_values,duplicate_gps,non_increasing_timestamps,eligible_route_pairs,disconnected_route_pairs,route_connectivity_percent
0,train,00000001,1315,0,88809,0,165790,0,525,0,0,0,0,524,0,100.000000
1,train,00000002,7243,0,7936,0,16139,0,262,0,0,0,0,261,1,99.616858
2,train,00000003,821,0,9138,0,18028,0,79,0,0,0,0,78,0,100.000000
3,train,00000004,1064,0,7350,0,15499,0,320,0,0,0,0,319,0,100.000000
4,train,00000005,1045,0,898,0,1579,0,105,0,0,0,0,104,0,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,student_test,00000056,3958,0,14759,0,27631,0,1323,0,0,0,0,1322,0,100.000000
86,student_test,00000065,768,0,6448,0,13184,0,145,0,0,0,0,144,0,100.000000
87,student_test,00000076,1486,0,26635,0,56027,0,374,0,0,0,0,373,1,99.731903
88,student_test,00000077,3911,0,4424,0,9622,0,268,0,0,0,0,267,0,100.000000


In [24]:

# Identify trips with disconnected route pairs

problem_trips = trip_summary_df[
    trip_summary_df["disconnected_route_pairs"] > 0
]

for _, row in problem_trips.iterrows():

    partition = row["partition"]
    trip_id = row["trip_id"]

    print("\n" + "=" * 60)
    print("Trip:", trip_id)
    print("Partition:", partition)

    track, nodes, arcs, route = load_raw_trip(
        partition, trip_id
    )

    route_ids = route["arc_id"].astype(int).to_numpy()
    route_arcs = arcs.iloc[route_ids].to_numpy()

    # Find disconnected transitions
    disconnected_positions = np.where(
        route_arcs[:-1, 1] != route_arcs[1:, 0]
    )[0]

    for pos in disconnected_positions:

        print("\nDisconnected transition at route positions:",
              pos, "and", pos + 1)

        print("Previous arc ID:", route_ids[pos])
        print("Previous arc:", route_arcs[pos])

        print("Next arc ID:", route_ids[pos + 1])
        print("Next arc:", route_arcs[pos + 1])



Trip: 00000002
Partition: train

Disconnected transition at route positions: 53 and 54
Previous arc ID: 1848
Previous arc: [ 846 5059]
Next arc ID: 10863
Next arc: [5289 5290]

Trip: 00000019
Partition: train

Disconnected transition at route positions: 94 and 95
Previous arc ID: 85589
Previous arc: [40766   849]
Next arc ID: 51027
Next arc: [23488 25511]

Disconnected transition at route positions: 151 and 152
Previous arc ID: 55059
Previous arc: [25416 25413]
Next arc ID: 55050
Next arc: [25408   889]

Disconnected transition at route positions: 300 and 301
Previous arc ID: 953
Previous arc: [  434 10107]
Next arc ID: 23709
Next arc: [10862   435]

Trip: 00000033
Partition: train

Disconnected transition at route positions: 164 and 165
Previous arc ID: 1003
Previous arc: [ 563 1283]
Next arc ID: 1002
Next arc: [563 553]

Trip: 00000041
Partition: train

Disconnected transition at route positions: 12 and 13
Previous arc ID: 1264
Previous arc: [779 349]
Next arc ID: 519
Next arc: [348

In [25]:

from math import radians, sin, cos, sqrt, atan2

def haversine(lon1, lat1, lon2, lat2):
    R = 6371000

    lon1, lat1, lon2, lat2 = map(
        radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )

    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


problem_trips = trip_summary_df[
    trip_summary_df["disconnected_route_pairs"] > 0
]

gap_results = []

for _, row in problem_trips.iterrows():

    partition = row["partition"]
    trip_id = row["trip_id"]

    track, nodes, arcs, route = load_raw_trip(
        partition, trip_id
    )

    route_ids = route["arc_id"].astype(int).to_numpy()
    route_arcs = arcs.iloc[route_ids].to_numpy()

    disconnected = np.where(
        route_arcs[:-1, 1] != route_arcs[1:, 0]
    )[0]

    for pos in disconnected:

        previous_end = int(route_arcs[pos, 1])
        next_start = int(route_arcs[pos + 1, 0])

        lon1, lat1 = nodes.iloc[previous_end]
        lon2, lat2 = nodes.iloc[next_start]

        gap = haversine(lon1, lat1, lon2, lat2)

        gap_results.append({
            "trip_id": trip_id,
            "partition": partition,
            "route_position": pos,
            "previous_end_node": previous_end,
            "next_start_node": next_start,
            "geographical_gap_m": round(gap, 2)
        })

gap_df = pd.DataFrame(gap_results)

display(gap_df)


,trip_id,partition,route_position,previous_end_node,next_start_node,geographical_gap_m
0,00000002,train,53,5059,5289,324.68
1,00000019,train,94,849,23488,241.02
2,00000019,train,151,25413,25408,255.19
3,00000019,train,300,10107,10862,595.81
4,00000033,train,164,1283,563,91.93
5,00000041,train,12,349,348,57.23
6,00000042,train,1212,1762,8726,30.72
7,00000069,train,153,5083,5084,40.86
8,00000075,train,31,250,227,10.62
9,00000075,train,62,229,247,73.05


In [26]:
direct_connectivity_results = []

for partition, trip_ids in all_trips.items():

    for trip_id in trip_ids:

        track, nodes, arcs, route = load_raw_trip(
            partition, trip_id
        )

        # Get arc IDs from the reference route
        route_ids = route["arc_id"].to_numpy().astype(int)

        # Retrieve the corresponding arcs
        route_arcs = arcs.iloc[route_ids].to_numpy()

        # Compare the ending node of each arc
        # with the starting node of the next arc
        disconnected_positions = np.where(
            route_arcs[:-1, 1] != route_arcs[1:, 0]
        )[0]

        direct_connectivity_results.append({
            "partition": partition,
            "trip_id": trip_id,
            "total_route_entries": len(route),
            "total_transitions": max(len(route) - 1, 0),
            "disconnected_transitions": len(disconnected_positions)
        })

# Create diagnostic DataFrame
connectivity_check_df = pd.DataFrame(
    direct_connectivity_results
)

print("Total trips checked:", len(connectivity_check_df))
print(
    "Total disconnected transitions:",
    connectivity_check_df["disconnected_transitions"].sum()
)

display(
    connectivity_check_df[
        connectivity_check_df["disconnected_transitions"] > 0
    ]
)

Total trips checked: 90
Total disconnected transitions: 14


,partition,trip_id,total_route_entries,total_transitions,disconnected_transitions
1,train,00000002,262,261,1
14,train,00000019,545,544,3
26,train,00000033,169,168,1
33,train,00000041,340,339,1
34,train,00000042,1440,1439,1
55,train,00000069,467,466,1
60,train,00000075,108,107,2
63,train,00000080,139,138,1
69,train,00000086,671,670,1
71,train,00000089,265,264,1


In [27]:
comparison_df = connectivity_check_df.merge(
    trip_summary_df[
        ["partition", "trip_id", "disconnected_route_pairs"]
    ],
    on=["partition", "trip_id"],
    how="left"
)

comparison_df["counts_match"] = (
    comparison_df["disconnected_transitions"]
    ==
    comparison_df["disconnected_route_pairs"]
)

print("Do all connectivity counts match?")
print(comparison_df["counts_match"].all())

display(
    comparison_df[
        comparison_df["counts_match"] == False
    ]
)

Do all connectivity counts match?
True


,partition,trip_id,total_route_entries,total_transitions,disconnected_transitions,disconnected_route_pairs,counts_match


In [28]:
# Overall dataset validation report

validation_report = {
    "total_trips": len(trip_summary_df),

    "total_gps_observations":
        trip_summary_df["gps_observations"].sum(),

    "total_road_nodes":
        trip_summary_df["road_nodes"].sum(),

    "total_road_arcs":
        trip_summary_df["road_arcs"].sum(),

    "total_route_entries":
        trip_summary_df["route_entries"].sum(),

    "invalid_gps":
        trip_summary_df["invalid_gps"].sum(),

    "invalid_nodes":
        trip_summary_df["invalid_nodes"].sum(),

    "invalid_arcs":
        trip_summary_df["invalid_arcs"].sum(),

    "invalid_route_entries":
        trip_summary_df["invalid_route_entries"].sum(),

    "missing_values":
        trip_summary_df["missing_values"].sum(),

    "duplicate_gps":
        trip_summary_df["duplicate_gps"].sum(),

    "non_increasing_timestamps":
        trip_summary_df["non_increasing_timestamps"].sum(),

    "disconnected_route_transitions":
        connectivity_check_df["disconnected_transitions"].sum()
}

report_df = pd.DataFrame(
    validation_report.items(),
    columns=["Validation Metric", "Value"]
)

display(report_df)

,Validation Metric,Value
0,total_trips,90
1,total_gps_observations,227577
2,total_road_nodes,2024174
3,total_road_arcs,4061945
4,total_route_entries,34813
5,invalid_gps,0
6,invalid_nodes,0
7,invalid_arcs,0
8,invalid_route_entries,0
9,missing_values,0


In [29]:
trip_summary_df.to_csv(
    "PathSnap_Trip_Validation_Report.csv",
    index=False
)

connectivity_check_df.to_csv(
    "PathSnap_Connectivity_Report.csv",
    index=False
)

report_df.to_csv(
    "PathSnap_Overall_Preprocessing_Report.csv",
    index=False
)

print("All preprocessing reports saved successfully!")

All preprocessing reports saved successfully!
